# Day 5 · Notebook 3 of 5 – UAT sign-off

**What this notebook does:** answers 12 business questions in two independent ways and saves the result to `capstone_5iot.gold.uat_signoff`.

* **expected_result** – calculated straight from the Gold/Silver base tables (a different query path), or a figure given by the client (3,349 min, 665 failures, 150 machines).
* **actual_result** – what the dashboard view shows.
* **status** – PASS when both agree.

After the client walkthrough, fill in `tester`, `signed_off_by` and `signoff_ts` (see the last cell). Re-running the notebook refreshes the results but **keeps** any names already entered.

**Run after** `7_1_dashboard_views`.

## Setup

In [0]:
from pyspark.sql import functions as F

GOLD, SILVER = "capstone_5iot.gold", "capstone_5iot.silver"
FME = f"{GOLD}.fact_maintenance_episodes"
ABNORMAL = "(temperature_c > 85 OR vibration_mm_s > 7.1 OR pressure_bar > 18)"

uat_rows = []


def q(sql):
    return spark.sql(sql).first()[0]


def view(name):
    return spark.table(f"{GOLD}.{name}")


def uat(uat_id, question, expected, actual):
    """Store one scenario. expected and actual are already formatted as text, so they are compared as text."""
    status = "PASS" if expected == actual else "FAIL"
    uat_rows.append((uat_id, question, expected, actual, status))
    print(f"{status}  {uat_id}  {question}\n       expected: {expected}\n       actual:   {actual}")


headline = view("vw_kpi_headline").first()
MACHINES = spark.table(f"{GOLD}.dim_machine").count()
DAYS     = spark.table(f"{GOLD}.dim_date").count()

## The 12 scenarios

In [0]:
# UAT-01  overall availability
downtime = q(f"SELECT sum(downtime_minutes) FROM {FME}")
expected = 1 - downtime / (MACHINES * DAYS * 1440)
uat("UAT-01", "What is overall equipment availability for the period?",
    f"{expected:.2%}", f"{headline.overall_availability:.2%}")

# UAT-02 / 03 / 04  client figures
uat("UAT-02", "What is total maintenance downtime (minutes)?", "3,349 min", f"{headline.total_downtime_minutes:,} min")
uat("UAT-03", "How many failures were recorded?", "665", str(headline.failure_count))
uat("UAT-04", "How many machines are monitored?", "150", str(headline.machines_monitored))

In [0]:
# UAT-05  top 5 machines by downtime
top_expected = spark.sql(f"""SELECT machine_id, sum(downtime_minutes) AS d FROM {FME}
                              GROUP BY machine_id ORDER BY d DESC, machine_id LIMIT 5""").collect()
top_actual = view("vw_machine_reliability").orderBy(F.desc("downtime_minutes"), "machine_id").limit(5).collect()
uat("UAT-05", "Which 5 machines had the most downtime?",
    ", ".join(f"{r.machine_id} ({r.d} min)" for r in top_expected),
    ", ".join(f"{r.machine_id} ({r.downtime_minutes} min)" for r in top_actual))

# UAT-06  failure codes that cause 80% of downtime (expected: own running total in Python)
codes = spark.sql(f"""SELECT failure_code, sum(downtime_minutes) AS d, count(*) AS n FROM {FME}
                       GROUP BY failure_code ORDER BY d DESC, n DESC, failure_code""").collect()
total, running, vital_expected = sum(r.d for r in codes), 0, []
for r in codes:
    if running < 0.8 * total:          # still below 80% before this code -> it belongs to the vital few
        vital_expected.append(r.failure_code)
    running += r.d
vital_actual = [r.failure_code for r in view("vw_failure_pareto").filter("pareto_class LIKE 'Vital few%'")
                                                                 .orderBy("pareto_rank").collect()]
uat("UAT-06", "Which failure codes cause 80% of downtime?",
    f"{len(vital_expected)} codes: {', '.join(vital_expected)}", f"{len(vital_actual)} codes: {', '.join(vital_actual)}")

In [0]:
# UAT-07  are abnormal readings higher in the 24h before a failure? (expected: from Silver tables)
rates = spark.sql(f"""
  WITH before_failure AS (
         SELECT DISTINCT r.reading_id
         FROM {SILVER}.silver_sensor_readings r JOIN {SILVER}.silver_failure_events f
           ON r.machine_id = f.machine_id AND r.reading_ts <= f.failure_ts AND r.reading_ts > f.failure_ts - INTERVAL 24 HOURS),
       no_failure AS (
         SELECT r.* FROM {SILVER}.silver_sensor_readings r
         LEFT ANTI JOIN {SILVER}.silver_failure_events f ON r.machine_id = f.machine_id)
  SELECT (SELECT round(avg(CAST(is_abnormal AS INT)), 4) FROM {SILVER}.silver_sensor_readings
           WHERE reading_id IN (SELECT reading_id FROM before_failure)) AS before_rate,
         (SELECT round(avg(CAST(is_abnormal AS INT)), 4) FROM no_failure) AS no_failure_rate""").first()
groups = {r.reading_group[:2]: r.abnormal_reading_rate for r in view("vw_failure_correlation").collect()}

def answer(before, none):
    return f"{'Yes' if before > none else 'No'}: {before:.2%} before failures vs {none:.2%} on machines with no failure"

uat("UAT-07", "Are abnormal readings higher in the 24h before failures than on machines with no failure?",
    answer(rates.before_rate, rates.no_failure_rate), answer(groups["1."], groups["3."]))

# UAT-08  quarantined records and why
expected = spark.sql(f"SELECT rule_id, count(*) AS n FROM {SILVER}.silver_rejected_records GROUP BY rule_id ORDER BY rule_id").collect()
actual = view("vw_data_quality").filter("metric_group = 'rejected_by_rule'").orderBy("metric_name").collect()
uat("UAT-08", "How many records were quarantined and why?",
    "; ".join(f"{r.rule_id} = {r.n}" for r in expected),
    "; ".join(f"{r.metric_name} = {int(r.metric_value)}" for r in actual))

# UAT-09  abnormal reading rate
expected = q(f"SELECT count_if({ABNORMAL}) / count(*) FROM {SILVER}.silver_sensor_readings")
uat("UAT-09", "What share of valid readings are abnormal (temp > 85, vibration > 7.1, pressure > 18)?",
    f"{expected:.2%}", f"{headline.abnormal_reading_rate:.2%}")

In [0]:
# UAT-10  region with most downtime
expected = spark.sql(f"""SELECT p.region, sum(f.downtime_minutes) AS d
                          FROM {FME} f JOIN {GOLD}.dim_plant p ON f.plant_sk = p.plant_sk
                          GROUP BY p.region ORDER BY d DESC, p.region LIMIT 1""").first()
actual = (view("vw_plant_summary").groupBy("region").agg(F.sum("downtime_minutes").alias("d"))
                                   .orderBy(F.desc("d"), "region").first())
uat("UAT-10", "Which region has the most downtime?", f"{expected.region} ({expected.d:,} min)", f"{actual.region} ({actual.d:,} min)")

# UAT-11  daily trend adds up to the period totals
t = view("vw_daily_trend").agg(F.sum("downtime_minutes").alias("d"), F.sum("failures").alias("f"),
                               F.sum("valid_readings").alias("r")).first()
uat("UAT-11", "Does the daily trend add up to the period totals (downtime, failures, readings)?",
    "3,349 min, 665 failures, 11,566 readings", f"{t.d:,} min, {t.f} failures, {t.r:,} readings")

# UAT-12  open maintenance backlog
expected = spark.sql(f"""SELECT count(*) AS n, sum(w.parts_cost) AS cost
                          FROM {SILVER}.silver_work_order_current c
                          JOIN {SILVER}.silver_work_orders w ON w.work_order_id = c.work_order_id
                          WHERE c.status <> 'COMPLETED'""").first()
actual = (view("vw_maintenance").filter("work_order_status <> 'COMPLETED'")
                                 .agg(F.sum("work_orders").alias("n"), F.sum("total_parts_cost").alias("cost")).first())
uat("UAT-12", "How many work orders are still open or in progress, and what parts cost do they carry?",
    f"{expected.n} work orders, parts cost {expected.cost:,.0f}", f"{actual.n} work orders, parts cost {actual.cost:,.0f}")

## Save to `gold.uat_signoff`
The MERGE updates the results but never overwrites tester / sign-off names that were already filled in.

In [0]:
UAT = f"{GOLD}.uat_signoff"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {UAT} (
    uat_id STRING, business_question STRING, expected_result STRING, actual_result STRING, status STRING,
    tester STRING, signed_off_by STRING, signoff_ts TIMESTAMP)
  COMMENT 'Day 5 UAT: business questions, expected (independent) vs actual (dashboard) results'""")

(spark.createDataFrame(uat_rows, "uat_id string, business_question string, expected_result string, actual_result string, status string")
      .createOrReplaceTempView("uat_new"))

spark.sql(f"""
MERGE INTO {UAT} t USING uat_new s ON t.uat_id = s.uat_id
WHEN MATCHED THEN UPDATE SET
     business_question = s.business_question, expected_result = s.expected_result,
     actual_result = s.actual_result, status = s.status
WHEN NOT MATCHED THEN INSERT (uat_id, business_question, expected_result, actual_result, status, tester, signed_off_by, signoff_ts)
     VALUES (s.uat_id, s.business_question, s.expected_result, s.actual_result, s.status,
             '<tester name>', '<client signatory>', NULL)""")

result = spark.table(UAT).orderBy("uat_id")
print("UAT:", result.filter("status = 'PASS'").count(), "of", result.count(), "scenarios PASS")
display(result)

## After the client walkthrough – record the sign-off
Fill in the names and run this cell once. Leave it commented out until then.

In [0]:
# spark.sql("""
# UPDATE capstone_5iot.gold.uat_signoff
# SET tester = 'Your Name', signed_off_by = 'Client Name', signoff_ts = current_timestamp()
# WHERE status = 'PASS'
# """)
# display(spark.table("capstone_5iot.gold.uat_signoff").orderBy("uat_id"))